# Tropical cyclone cylindrical transform

Runnable smoke notebook for the Fortran-backed xvortices core.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import xarray as xr

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from xvortices import load_cylind, project_to_cylind, storm_relative

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ds = xr.open_dataset(root / 'Data' / 'Haima2004.nc')
ds = ds[['u', 'v', 'w', 'h']].isel(lev=slice(0, 8))
ds

In [ ]:
times = ds.time
olon = xr.DataArray(np.linspace(121.0, 131.5, times.size), dims=('time',), coords={'time': times}, name='olon')
olat = xr.DataArray(np.linspace(17.0, 25.0, times.size), dims=('time',), coords={'time': times}, name='olat')

fields, lons, lats, etas = load_cylind(ds, olon=olon, olat=olat, azimNum=72, radiNum=31, radMax=6)
u, v, w, h = fields
assert u.dims == ('time', 'lev', 'radi', 'azim')
assert lons.dims == ('time', 'radi', 'azim')
float(u.mean())

In [ ]:
uaz, vra = project_to_cylind(u, v, etas)
uc = xr.DataArray(np.gradient(olon.values), dims=('time',), coords={'time': times})
vc = xr.DataArray(np.gradient(olat.values), dims=('time',), coords={'time': times})
uaz_rel, vra_rel = storm_relative(uc, vc, uaz, vra)

summary = xr.Dataset({
    'azimuthal_mean_wind': uaz_rel.mean(('time', 'azim')),
    'radial_mean_wind': vra_rel.mean(('time', 'azim')),
    'height_anomaly': (h - h.mean('azim')).mean('time'),
})
summary